# Module 3: Fine-tuning with QLoRA

**Duration:** ~60 minutes (including ~20 min training time)

## Learning Objectives
- Understand LoRA and QLoRA concepts
- Configure LoRA adapters for efficient training
- Prepare training data in instruction format
- Train the model using Unsloth for 2x speedup
- Save and test the fine-tuned model

## Overview
Fine-tuning teaches the model F5-specific knowledge by training on Q&A pairs. We use QLoRA (Quantized LoRA) to enable training on limited GPU memory while maintaining quality.

## 3.1 Understanding LoRA

**LoRA (Low-Rank Adaptation)** enables efficient fine-tuning by:
- Freezing the original model weights
- Adding small trainable "adapter" matrices
- Training only 0.1-1% of parameters

**QLoRA** adds 4-bit quantization, reducing memory requirements even further.

```
Original Model      LoRA Adapters
┌───────────┐       ┌───┐
│  Frozen   │   +   │ A │ ← Trainable
│  Weights  │       │ B │ ← Trainable  
│  (4-bit)  │       └───┘
└───────────┘       (< 1% params)
```

## 3.2 Install Unsloth

Unsloth provides optimized training that's 2x faster than standard implementations.

In [ ]:
%%capture
# Install Unsloth for Colab
!pip uninstall unsloth -y 2>/dev/null
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install other dependencies
!pip install -q trl>=0.24.0 datasets

In [ ]:
import torch
import os

# Verify GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")
print(f"CUDA version: {torch.version.cuda}")

# Clone repo if running in Colab and data doesn't exist
if not os.path.exists('data/training'):
    if os.path.exists('llm-finetuning-rag-lab'):
        os.chdir('llm-finetuning-rag-lab')
    else:
        print("Cloning repository...")
        !git clone https://github.com/therealnoof/llm-finetuning-rag-lab.git
        os.chdir('llm-finetuning-rag-lab')
        print("✅ Repository cloned!")

print(f"Working directory: {os.getcwd()}")

# Verify data exists
if os.path.exists('data/training'):
    print(f"✅ Training data directory found")
else:
    print("❌ Training data not found - check repository clone")

## 3.3 Load Model with Unsloth

Unsloth provides an optimized model loader that works with their training pipeline.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LENGTH = 2048

print(f"Loading {MODEL_NAME} with Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True  # QLoRA
)

print("✅ Model loaded with Unsloth optimization")

In [ ]:
# Check memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"GPU Memory allocated: {allocated:.2f} GB")

## 3.4 Configure LoRA Adapters

We'll add LoRA adapters to all linear layers for maximum learning capacity while keeping training efficient.

In [ ]:
# =============================================================================
# WHY THESE LoRA SETTINGS?
# =============================================================================
# LoRA adds small trainable matrices to frozen model weights. Key parameters:
#
# r (rank): Controls adapter capacity. Higher = more learning capacity but
#           more parameters. 16 is a good balance for domain adaptation.
#           Common values: 8 (light), 16 (balanced), 32+ (heavy adaptation)
#
# lora_alpha: Scaling factor for LoRA updates. Usually set equal to r.
#             Higher values = stronger LoRA influence on outputs.
#
# target_modules: Which layers to add adapters to. We target:
#   - Attention layers (q,k,v,o_proj): How the model "pays attention"
#   - MLP layers (gate,up,down_proj): The model's "reasoning" capacity
#   Targeting all linear layers gives maximum adaptation capability.
#
# use_gradient_checkpointing: Trades compute for memory - recomputes 
#                             activations during backward pass instead of storing them.
# =============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # LoRA rank - 16 is good for domain adaptation
    lora_alpha=16,             # Scaling factor - typically equal to r
    lora_dropout=0.0,          # No dropout = faster training (Unsloth recommendation)
    target_modules=[
        # Attention layers - how the model focuses on relevant information
        "q_proj", "k_proj", "v_proj", "o_proj",
        # MLP layers - the model's feedforward reasoning capacity  
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",               # Don't train biases - reduces parameters
    use_gradient_checkpointing="unsloth",  # Memory optimization (2x less VRAM)
    random_state=42,           # Reproducibility
    use_rslora=False,          # Standard LoRA (RSLoRA is experimental)
    loftq_config=None          # No quantization-aware initialization
)

print("✅ LoRA adapters configured")

In [ ]:
# Show trainable parameters
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable, total = count_parameters(model)
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Percentage trainable: {100 * trainable / total:.2f}%")

## 3.5 Prepare Training Data

We'll load our F5 Q&A pairs and format them for training.

In [ ]:
from datasets import load_dataset

# Load training data
dataset = load_dataset(
    "json",
    data_files="data/training/f5_qa_train.jsonl",
    split="train"
)

print(f"✅ Loaded {len(dataset)} training examples")
print(f"\nSample entry:")
print(f"  Question: {dataset[0]['question'][:80]}...")
print(f"  Answer: {dataset[0]['answer'][:80]}...")

In [ ]:
# =============================================================================
# WHY THIS SPECIFIC CHAT TEMPLATE?
# =============================================================================
# Each LLM has a specific format it was pre-trained with. Using the wrong
# format leads to poor results. TinyLlama uses this structure:
#   <|system|> ... </s>  - Sets the AI's persona/role
#   <|user|> ... </s>    - The human's input
#   <|assistant|> ... </s> - The AI's response (what we're training it to generate)
#
# The </s> token marks the end of each turn. This helps the model understand
# conversation structure and know when to stop generating.
#
# IMPORTANT: Always match the template used during pre-training/instruction-tuning
# of your base model. Check the model card on HuggingFace for the correct format.
# =============================================================================

CHAT_TEMPLATE = """<|system|>
You are an F5 Networks technical expert. Provide accurate, detailed answers about BIG-IP, iRules, load balancing, SSL offloading, and other F5 technologies.</s>
<|user|>
{question}</s>
<|assistant|>
{answer}</s>"""

def format_example(example):
    """
    Format a single Q&A pair into the chat template.
    
    This converts our simple {"question": ..., "answer": ...} format
    into the full chat format the model expects during training.
    """
    text = CHAT_TEMPLATE.format(
        question=example["question"],
        answer=example["answer"]
    )
    return {"text": text}

# Apply formatting to all examples in the dataset
# remove_columns drops the original columns, keeping only the formatted "text"
formatted_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

print("✅ Dataset formatted")
print(f"\nSample formatted text:\n{formatted_dataset[0]['text'][:500]}...")

## 3.6 Configure Training

We'll use the TRL library's SFTTrainer (Supervised Fine-Tuning Trainer) for training.

In [ ]:
from trl import SFTTrainer, SFTConfig

# =============================================================================
# WHY SFTTrainer (Supervised Fine-Tuning Trainer)?
# =============================================================================
# SFTTrainer from the TRL (Transformer Reinforcement Learning) library is 
# specifically designed for instruction-tuning LLMs. Unlike the base HuggingFace
# Trainer, SFTTrainer:
#   1. Handles chat/instruction formatting automatically
#   2. Supports packing multiple examples into one sequence (efficiency)
#   3. Integrates seamlessly with PEFT/LoRA adapters
#   4. Optimized for the supervised fine-tuning phase of LLM training
#
# The training flow for modern LLMs is typically:
#   Pre-training → SFT (we are here) → RLHF (optional)
# =============================================================================

training_args = SFTConfig(
    output_dir="./outputs",
    
    # -------------------------------------------------------------------------
    # EPOCHS: How many times to iterate over the entire dataset
    # 3 epochs is a good starting point - too few = underfit, too many = overfit
    # -------------------------------------------------------------------------
    num_train_epochs=3,
    max_steps=-1,  # -1 means use epochs instead of fixed steps
    
    # -------------------------------------------------------------------------
    # BATCH SIZE: How many examples to process at once
    # Larger batches = more stable gradients but need more memory
    # We use gradient accumulation to simulate larger batches:
    #   Effective batch size = per_device_batch (2) × accumulation_steps (4) = 8
    # -------------------------------------------------------------------------
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    
    # -------------------------------------------------------------------------
    # LEARNING RATE: How big of steps to take during optimization
    # 2e-4 is a good default for LoRA fine-tuning (higher than full fine-tuning)
    # Warmup helps prevent early training instability
    # Cosine scheduler gradually reduces LR for smoother convergence
    # -------------------------------------------------------------------------
    learning_rate=2e-4,
    warmup_ratio=0.03,  # 3% of training steps for warmup
    lr_scheduler_type="cosine",
    
    # -------------------------------------------------------------------------
    # OPTIMIZATION: AdamW with 8-bit precision saves memory
    # Weight decay (0.01) helps prevent overfitting by penalizing large weights
    # Gradient clipping (0.3) prevents exploding gradients
    # -------------------------------------------------------------------------
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    
    # Logging - track progress every 10 steps
    logging_steps=10,
    logging_first_step=True,
    
    # Saving - checkpoint every 100 steps, keep last 2
    save_steps=100,
    save_total_limit=2,
    
    # -------------------------------------------------------------------------
    # PRECISION: Use bf16 if available (better for training), else fp16
    # Mixed precision training is faster and uses less memory
    # -------------------------------------------------------------------------
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Other settings
    seed=42,  # For reproducibility
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # Don't pack multiple examples (simpler for learning)
    dataset_text_field="text"  # Column name containing formatted text
)

print("✅ Training configuration set")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    args=training_args
)

print("✅ Trainer created")

## 3.7 Train the Model

Training will take approximately <5 minutes on a T4 GPU with our dataset size.

Watch the loss decrease - this indicates the model is learning!

In [ ]:
# Record GPU memory before training
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    start_memory = torch.cuda.memory_allocated() / 1e9
    print(f"GPU Memory before training: {start_memory:.2f} GB")

print("\n🚀 Starting training...")
print("This will take approximately 15-20 minutes.")
print("-" * 50)

In [ ]:
# Train!
trainer_stats = trainer.train()

print("\n✅ Training complete!")

In [ ]:
# Training statistics
print("\n📊 Training Statistics:")
print(f"   Total steps: {trainer_stats.global_step}")
print(f"   Training loss: {trainer_stats.training_loss:.4f}")
print(f"   Training time: {trainer_stats.metrics.get('train_runtime', 0) / 60:.1f} minutes")

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1e9
    print(f"   Peak GPU memory: {peak_memory:.2f} GB")

## 3.8 Save the Fine-tuned Model

We'll save just the LoRA adapters (small) rather than the full model.

In [ ]:
# Save the LoRA adapters
SAVE_PATH = "./outputs/f5_assistant_lora"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Model saved to {SAVE_PATH}")

# Check saved files
!ls -lh {SAVE_PATH}

## 3.9 Test the Fine-tuned Model

Let's test our fine-tuned model on the evaluation questions.

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def generate_response(question, max_new_tokens=256, temperature=0.7):
    """Generate a response from the fine-tuned model."""
    prompt = f"""<|system|>
You are an F5 Networks technical expert. Provide accurate, detailed answers about BIG-IP, iRules, load balancing, SSL offloading, and other F5 technologies.</s>
<|user|>
{question}</s>
<|assistant|>
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()
    
    return response

print("✅ Inference mode enabled")

In [ ]:
# Quick test
test_question = "What is a virtual server in F5 BIG-IP?"
print(f"Question: {test_question}\n")
print(f"Response:\n{generate_response(test_question)}")

In [ ]:
# Evaluate on all questions
eval_questions = [
    "What is a virtual server in F5 BIG-IP and what is its purpose?",
    "How do I configure SSL offloading on F5 BIG-IP?",
    "Write an iRule to redirect HTTP to HTTPS.",
    "What is the difference between Least Connections and Round Robin load balancing?",
    "How do I troubleshoot a pool member that is marked as offline?"
]

print(f"Evaluating {len(eval_questions)} questions...\n")
print("=" * 80)

In [ ]:
# Store fine-tuned responses
finetuned_responses = []

for i, question in enumerate(eval_questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {question}")
    print("-" * 80)
    
    response = generate_response(question)
    finetuned_responses.append({
        "question": question,
        "response": response
    })
    
    print(f"Response:\n{response}")

print(f"\n{'='*80}")
print("Fine-tuned model evaluation complete!")

## 3.10 Save Fine-tuned Results

In [ ]:
import json
import os

os.makedirs("results", exist_ok=True)

# Save results
results = {
    "model": MODEL_NAME,
    "type": "finetuned",
    "lora_config": {
        "r": 16,
        "lora_alpha": 16,
        "epochs": 3
    },
    "training_loss": trainer_stats.training_loss,
    "responses": finetuned_responses
}

with open("results/finetuned_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Fine-tuned results saved to results/finetuned_results.json")

## 3.11 Understanding Fine-tuning Trade-offs

### Advantages of Fine-tuning
✅ **Learns domain patterns** - Model internalizes knowledge  
✅ **Faster inference** - No retrieval step needed  
✅ **Captures implicit knowledge** - Learns writing style, terminology  
✅ **Better generalization** - Can apply patterns to new questions  

### Limitations
⚠️ **Requires training data** - Need quality Q&A pairs  
⚠️ **Risk of overfitting** - May memorize instead of learn  
⚠️ **Harder to update** - Need to retrain for new information  
⚠️ **Catastrophic forgetting** - May lose general knowledge  

### When to Use Fine-tuning
- ✅ Consistent domain-specific terminology needed
- ✅ Have substantial training data (100+ examples)
- ✅ Need fast inference without retrieval
- ✅ Want to capture reasoning patterns

## 3.12 Summary

In this module, we:

1. ✅ Loaded TinyLlama with Unsloth for optimized training
2. ✅ Configured LoRA adapters (r=16, all linear layers)
3. ✅ Prepared training data in chat format
4. ✅ Trained for 3 epochs on F5 Q&A data
5. ✅ Saved and tested the fine-tuned model

### Key Observations
- Training loss decreased, indicating learning
- Fine-tuned responses use F5-specific terminology
- Model can generate iRules and TMSH commands

### Next Steps
In **Module 4**, we'll compare all three approaches side-by-side and evaluate which performs best for different types of questions.

In [ ]:
# Cleanup - free GPU memory before Module 4
import gc
del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()
print("✅ GPU memory cleared - ready for Module 4")